### Exit Ticket Questions

* How can we search all the lessons when trying to find something?
    - Trello board organized thematically
* Should you always use elastic net?
    - Takes longer to fit
    - Lasso can offer a more sparse representation; Elastic Net is likely to retain more features (some number > Lasso and < Ridge)
    - If you only care about optimal performance, maybe?

* Sample predictive modelling workflow:
    - Data Cleaning
    - Splitting the data into training and testing sets. 
        * We don't want our test data to learn anything from the training data
    - Preprocessing: Standardization, PCA, other transformations
        * Fit/transform on training data
        * Transform only on testing data
    - Validating each potential predictors relationship to the target or deriving new predictors with some relationship to the target (EDA)
    - Tuning feature selection; model choice and parameters using cross validation
    - Selecting regressors or classifiers to fit and tune with cross validation
    - Fitting your best set of features in your best performing model and then predicting and scoring on your test data
    - Analyze your error and mitigate it (by e.g. deriving new features)

# Capstone Requirements

Your presentation should include:
* A problem statement.
* Description of and assumptions about the data
* Discussion of your features and feature importance
* An explanation of your cost metric and why it was chosen
* The performance of different models you scored
* Your best performing model performance relative to some baseline
* **Encouraged:** Details of the error analysis isolating strengths and shortcomings you performed on your model  
* Recommendations or next steps.

In [ ]:
import pandas as pd
import numpy as np
import scipy as sp
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
#from textblob import TextBlob, Word
from nltk.stem.snowball import SnowballStemmer

%matplotlib inline

## What Is Natural Language Processing (NLP)?

- Using computers to process (analyze, understand, generate) natural human languages.

### Higher-Level NLP Task Areas

Some higher-level tasks include:

- **Question answering:** Determine the intent of the question, match query with knowledge base, evaluate hypotheses.
- **Information retrieval:** Find relevant results and similar results.
- **Machine translation:** One language to another.
- **Predictive text input:** Faster or easier typing.

### Higher-Level Tasks Are Hard Because

- **Ambiguity**:
    - Hospitals Are Sued by 7 Foot Doctors
    - Juvenile Court to Try Shooting Defendant
    - Local High School Dropouts Cut in Half
- **Idioms:** "throw in the towel"
- **Newly coined words & non-standard words:** "retweet"
- **Tricky entity names:** "Where is A Bug's Life playing?"

### Our Use Case: Text Classification
** the task of predicting which category or topic a text sample is from **

In [ ]:
# Read yelp.csv into a DataFrame.
path = r'./data/yelp.csv'
yelp = pd.read_csv(path)
yelp.head(1)

In [ ]:
#sample 'document'
yelp.text.values[0]

In text classification, we vectorize the text into a set of numeric features. 

- For a given document, the numeric value of each feature could be the number of times the word appears in the document.
    - So, most features ['tokens'] will have a value of zero, resulting in a sparse matrix of features.

- This technique for vectorizing text is referred to as a bag-of-words model. 
    - It is called bag of words because the document's structure is lost — as if the words are all jumbled up in a bag.
    
Now we can apply a machine learning classifier.

![DTM](images/DTM.png)

## Preprocessing 
** Before we vectorize (or 'tokenize') our text, we may use a variety of preprocessing tools **


### N-grams

N-grams are features which consist of N consecutive words. This is useful because using the bag-of-words model, treating `data scientist` as a single feature has more meaning than having two independent features `data` and `scientist`!

Example:
```
my cat is awesome
Unigrams (1-grams): 'my', 'cat', 'is', 'awesome'
Bigrams (2-grams): 'my cat', 'cat is', 'is awesome'
Trigrams (3-grams): 'my cat is', 'cat is awesome'
4-grams: 'my cat is awesome'
```
- You set the lower and upper boundary of the range of n-values for different n-grams to be extracted. All values of n such that min_n <= n <= max_n will be used.

### Stop-Word Removal

Stop words are some of the most common words in a language e.g., "to," "the," "and." However, they are so commonly used that they are generally worthless for predicting the class of a document. 

### Capitalization

- Does upper or lower case matter? to a computer, aPPle is a different 'token' than apple

### Setting a cut-off for occurrences
- Extrapolating from infrequently occurring terms can lead to overfitting. When building the vocabulary, ignore terms that have a document frequency strictly lower than the given threshold. 


### Stemming and Lemmatization

Stemming is a crude process of removing common endings from sentences, such as "s", "es", "ly", "ing", and "ed".

- **What:** Reduce a word to its base/stem/root form.
- **Why:** This intelligently reduces the number of features by grouping together (hopefully) related words.

Lemmatization is a more refined process that uses specific language and grammar rules to derive the root of a word.  

This is useful for words that do not share an obvious root such as "better" and "best".

- **What:** Lemmatization derives the canonical form ("lemma") of a word.

**More Lemmatization and Stemming Examples**

|Lemmatization|Stemming|
|-------------|---------|
|shouted → shout|badly → bad|
|best → good|computing → comput|
|better → good|computed → comput|
|good → good|wipes → wip|
|wiping → wipe|wiped → wip|
|hidden → hide|wiping → wip|

In [ ]:
# Create a new DataFrame that only contains the 5-star and 1-star reviews.
yelp_best_worst = yelp[(yelp.stars==5) | (yelp.stars==1)]

# Define X and y.
X = yelp_best_worst.text
y = yelp_best_worst.stars

# Split the new DataFrame into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(X, y)

# Use CountVectorizer to create document-term matrices from X_train and X_test. Note parameters.
vect = CountVectorizer(lowercase=True,min_df=2,ngram_range=(1,2),
                       stop_words='english')

# fit: Learn a vocabulary dictionary of all tokens in the raw documents.
# transform: Transform documents to document-term matrix.

# for train we fit *and* transform
X_train_dtm = vect.fit_transform(X_train)
# for test, we transform; we only use vocabulary present in train
X_test_dtm = vect.transform(X_test)
X_train_dtm.shape

In [ ]:
# We can access the tokens in our vocabulary using the get feature names method for the CountVectorizer object
print((vect.get_feature_names()[-50:]))

In [ ]:
# our transformed training data is a big pivot table stored in a memory efficient format. We can convert it to an 
# array or df but its memory allocation would be very expensive
type(X_train_dtm)

In [ ]:
# Use Logistic Regression to predict the star rating.

# instantiate
logr = LogisticRegression()
# fit
logr.fit(X_train_dtm, y_train)
# predict and store predictions
y_pred_class = logr.predict(X_test_dtm)
print('null_model: '+str(y_test.value_counts('mean').values[0]))
# Calculate accuracy.
print('model accuracy: '+str((accuracy_score(y_test, y_pred_class))))

In [ ]:
# it's a logistic regression, so we can return coefficients that indicate words that predict positive review
features = np.array(vect.get_feature_names())
logr_coefs = pd.DataFrame({'coef':logr.coef_[0]},index=features)
logr_coefs = logr_coefs.sort_values('coef',ascending=False)
logr_coefs.head(10)

In [ ]:
# as well as words that predict negative reviews
logr_coefs.tail(10)

## Term Frequency–Inverse Document Frequency (TF–IDF)

Term frequency–inverse document frequency (TF–IDF) computes the "relative frequency" with which a word appears in a document, compared to its frequency across all documents.

It's more useful than simple "term frequency" [the count vectorizer above] for identifying "important" words in each document (high frequency in that document, low frequency in other documents). (But not necessarily more performant.)

### Example 

In [ ]:
simple_train = ['call you tonight', 'Call me a cab', 'please call me... PLEASE!']

# Term frequency
vect = CountVectorizer()
tf = pd.DataFrame(vect.fit_transform(simple_train).toarray(), columns=vect.get_feature_names())
tf

In [ ]:
# Document frequency
vect = CountVectorizer(binary=True)
df = vect.fit_transform(simple_train).toarray().sum(axis=0)
pd.DataFrame(df.reshape(1, 6), columns=vect.get_feature_names())

In [ ]:
# Term frequency–inverse document frequency (simple version)
tf/df

In [ ]:
# TfidfVectorizer
vect = TfidfVectorizer()
pd.DataFrame(vect.fit_transform(simple_train).toarray(), columns=vect.get_feature_names())

## Pipelines

In [ ]:
# Pipeline of transforms with a final estimator. Sequentially apply a list of transforms and a final estimator. 
# Intermediate steps of the pipeline must be ‘transforms’, that is, they must implement fit and transform methods. 
# The final estimator only needs to implement fit.

from sklearn.pipeline import make_pipeline
model = make_pipeline(TfidfVectorizer(lowercase=True,
                                      max_df=1.0, 
                                      min_df=2,
                                      ngram_range=(1,2),
                                      stop_words='english'),
                      LogisticRegression())
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

## Sentiment

In [ ]:
tweets = pd.read_csv("./data/Tweets.csv",encoding = "ISO-8859-1")
tweets.head()

In [ ]:
tweets.airline_sentiment.value_counts()

In [ ]:
# install vaderSentiment first
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

### Sentiment ratings (via https://github.com/cjhutto/vaderSentiment)
Sentiment ratings from 10 independent human raters. Over 9,000 token features were rated on a scale from "[–4] Extremely Negative" to "[4] Extremely Positive", with allowance for "[0] Neutral (or Neither, N/A)". We kept every lexical feature that had a non-zero mean rating, and whose standard deviation was less than 2.5 as determined by the aggregate of those ten independent raters. This left us with just over 7,500 lexical features with validated valence scores that indicated both the sentiment polarity (positive/negative), and the sentiment intensity on a scale from –4 to +4. For example, the word "okay" has a positive valence of 0.9, "good" is 1.9, and "great" is 3.1, whereas "horrible" is –2.5, the frowning emoticon :( is –2.2, and "sucks" and it's slang derivative "sux" are both –1.5.

### Scores
The compound score is computed by summing the valence scores of each word in the lexicon, adjusted according to the rules, and then normalized to be between -1 (most extreme negative) and +1 (most extreme positive). **This is the most useful metric if you want a single unidimensional measure of sentiment for a given sentence. Calling it a 'normalized, weighted composite score' is accurate.**

It is also useful for researchers who would like to set standardized thresholds for classifying sentences as either positive, neutral, or negative. Typical threshold values (used in the literature cited on this page) are:

* positive sentiment: compound score >= 0.05
* neutral sentiment: (compound score > -0.05) and (compound score < 0.05)
* negative sentiment: compound score <= -0.05

**The pos, neu, and neg scores are ratios for proportions of text that fall in each category** (so these should all add up to be 1... or close to it with float operation). These are the most useful metrics if you want multidimensional measures of sentiment for a given sentence.

In [ ]:
# get polarity score for example - output is a dictionary
print(tweets['text'][0])
example = sia.polarity_scores(tweets['text'][0])
example

In [ ]:
# get polarity scores for each observation and add each score to df
def unload_dict(df):
    polarity = sia.polarity_scores(df['text'])
    df['compound'] = polarity['compound']
    df['neg'] = polarity['neg']
    df['neu'] = polarity['neu']
    df['pos'] = polarity['pos']
    return df

tweets = tweets.apply(unload_dict, axis=1)
tweets.sample(3)

In [ ]:
# this is a very high compound score - note that this is a fallible process
tweets['text'][2728]